# Triển khai "Nghịch lý kho hàng & Tốc độ bán"

Đầu tiên đọc các bảng cần thiết và nối các bảng

In [8]:
import pandas as pd

df_inventory = pd.read_csv('../../data/inventory.csv')
df_products = pd.read_csv('../../data/products.csv')
df_items = pd.read_csv('../../data/order_items.csv')
df_orders = pd.read_csv('../../data/orders.csv')
df_items_with_date = pd.merge(df_items, df_orders[['order_id', 'order_date']], on='order_id', how='left')
df_items_with_date['order_date'] = pd.to_datetime(df_items_with_date['order_date'])




/var/folders/n6/fpv8j7h17fxb5lbxwtz4qbyr0000gn/T/ipykernel_60071/1875636232.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv('../../data/order_items.csv')


Xác định mốc thời gian 90 ngày gần nhất để tính tốc độ bán thực tế

In [10]:
last_date = df_items_with_date['order_date'].max()
start_date = last_date - pd.Timedelta(days=90)
df_recent_sales = df_items_with_date[df_items_with_date['order_date'] >= start_date]

Tính tổng lượng bán và tốc độ bán trung bình ngày (Velocity)

In [11]:
df_velocity = df_recent_sales.groupby('product_id')['quantity'].sum().reset_index()
df_velocity['avg_daily_sales'] = df_velocity['quantity'] / 90

In [12]:
df_final = pd.merge(df_inventory, df_velocity, on='product_id', how='left').fillna(0)
df_final = pd.merge(df_final, df_products[['product_id', 'product_name', 'category', 'cogs']], on='product_id', how='left')

Tính toán các chỉ số "Sức khỏe tồn kho" (Feature Engineering)
- Tỷ lệ bán hết (Sell-through Rate - STR): Đo lường hiệu quả bán hàng so với lượng tồn.
- Số ngày cung ứng (Days of Supply - DOS): Dự báo bao nhiêu ngày nữa thì hết hàng nếu tốc độ bán giữ nguyên.
- Vốn đọng (Dead Capital): Số tiền đang bị "chôn" trong kho.

In [13]:
# STR = Bán ra / (Bán ra + Tồn kho hiện tại)
df_final['sell_through_rate'] = (df_final['quantity'] / (df_final['quantity'] + df_final['stock_on_hand'] + 0.001)) * 100
# DOS = Tồn kho / Bán trung bình ngày (nếu bán ngày = 0 thì để 999 ngày)
df_final['days_of_supply'] = df_final.apply(lambda x: x['stock_on_hand'] / x['avg_daily_sales'] if x['avg_daily_sales'] > 0 else 999, axis=1)
# Vốn đọng = Tồn kho * Giá vốn
df_final['dead_capital'] = df_final['stock_on_hand'] * df_final['cogs']

# Xuất file để đưa vào Tableau
df_final.to_csv('inventory_insight_ready.csv', index=False)